# Task 4: General Health Query Chatbot (Prompt Engineering Based)

**DevelopersHub Corporation — AI/ML Engineering Internship**

## Problem Statement

People often turn to the internet with simple health questions ("why does my
throat hurt?", "is this medicine safe for my kid?") before they can see a
doctor. The goal of this notebook is to build a conversational assistant that
answers **general** health questions in a friendly, clear tone — using
**prompt engineering** rather than model fine-tuning — while staying safely
within the bounds of what a non-medical AI assistant should say.

## Objective

- Send user health questions to an LLM (OpenAI GPT-3.5, or a free
  open-source alternative via the Hugging Face Inference API).
- Use a carefully engineered system prompt so responses are friendly, clear,
  and appropriately cautious.
- Add **safety filters** that intercept emergencies and unsafe requests
  (e.g. medication dosage/overdose questions) *before* they ever reach the
  LLM, and redirect the user to real professional help.

## Tools

- `openai` Python SDK (GPT-3.5-turbo) **or**
- Hugging Face Inference API (free tier) running `Mistral-7B-Instruct`

## ⚠️ A note on running this notebook

This notebook was developed in a sandboxed environment whose network access
is restricted to a short allow-list of domains (PyPI, GitHub, etc.) — it
cannot reach `api.openai.com` or `huggingface.co`. So that the notebook still
**runs end-to-end and demonstrates the full logic**, it ships with
`DEMO_MODE = True`, which uses a small offline response generator instead of
a live API call.

**To get real LLM-generated answers:** set `DEMO_MODE = False` below, add
your `OPENAI_API_KEY` (or `HF_API_TOKEN`) as an environment variable, and
re-run the notebook on your own machine or Google Colab — no other code
changes are needed, since the real API-calling functions (`call_openai`,
`call_huggingface`) are already fully implemented below.


## 1. Configuration & System Prompt

In [1]:
import os
import re

# ---------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------
API_PROVIDER = "openai"      # "openai" or "huggingface"
DEMO_MODE = True             # Set to False once a real API key is configured

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
HF_API_TOKEN = os.environ.get("HF_API_TOKEN", "")
HF_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# ---------------------------------------------------------------
# System prompt — this is the core of the "prompt engineering" for this task
# ---------------------------------------------------------------
SYSTEM_PROMPT = """You are a friendly, knowledgeable general-health information assistant.

Rules you must always follow:
1. You are NOT a doctor. Never diagnose a condition or prescribe a treatment.
2. Explain things simply and warmly, like a helpful nurse friend, not a textbook.
3. Keep answers concise (3-6 sentences) unless the user asks for more detail.
4. Never give exact medication dosages, especially for controlled or
   potentially dangerous substances. Tell the user to confirm dosing with a
   pharmacist or doctor.
5. Always end with a gentle reminder to consult a licensed healthcare
   professional for anything personal or serious.
6. If a question describes a medical emergency, do not attempt to answer —
   tell the user to seek immediate emergency care.
"""

print("Configuration loaded.")
print(f"Provider: {API_PROVIDER} | Demo mode: {DEMO_MODE}")


Configuration loaded.
Provider: openai | Demo mode: True


## 2. Safety Filters

Per the task requirements, we never let unsafe queries reach the LLM at all.
Two categories are intercepted directly:

- **Emergencies** (chest pain, can't breathe, suicidal thoughts, etc.) → the
  user is told to seek immediate real-world help.
- **Unsafe dosage / overdose questions** → the bot declines and redirects to
  a pharmacist or doctor.

In [2]:
EMERGENCY_PATTERNS = [
    r"chest pain", r"can.?t breathe", r"difficulty breathing", r"severe bleeding",
    r"suicide", r"kill myself", r"end my life", r"overdose right now",
    r"stroke symptoms", r"unconscious", r"not breathing", r"heart attack",
]

UNSAFE_DOSAGE_PATTERNS = [
    r"how many (mg|milligrams|pills|tablets)", r"lethal dose",
    r"how (much|many) .* (overdose|to die|kill (myself|me))",
    r"overdose .* how (much|many)",
]

def safety_check(query: str) -> str:
    """Returns 'emergency', 'unsafe_dosage', or 'safe'."""
    q = query.lower()
    if any(re.search(p, q) for p in EMERGENCY_PATTERNS):
        return "emergency"
    if any(re.search(p, q) for p in UNSAFE_DOSAGE_PATTERNS):
        return "unsafe_dosage"
    return "safe"


EMERGENCY_RESPONSE = (
    "⚠️ This sounds like it could be a medical emergency. Please call your "
    "local emergency number or go to the nearest emergency room right away. "
    "I'm not able to help with emergencies — please seek immediate in-person care."
)

UNSAFE_DOSAGE_RESPONSE = (
    "I'm not able to give specific medication doses, especially where getting "
    "it wrong could be dangerous. Please check with a licensed pharmacist or "
    "doctor — they can advise safely based on your full medical history."
)


## 3. LLM-Calling Functions (Real Implementation)

In [3]:
def call_openai(query: str) -> str:
    """Send the query to OpenAI GPT-3.5-turbo with the engineered system prompt."""
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
        temperature=0.4,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


def call_huggingface(query: str) -> str:
    """Send the query to a free open-source model via the HF Inference API."""
    import requests
    headers = {"Authorization": f"Bearer {HF_API_TOKEN}"}
    prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\nUser question: {query} [/INST]"
    payload = {"inputs": prompt, "parameters": {"max_new_tokens": 250, "temperature": 0.4}}
    resp = requests.post(
        f"https://api-inference.huggingface.co/models/{HF_MODEL_ID}",
        headers=headers, json=payload, timeout=30,
    )
    resp.raise_for_status()
    return resp.json()[0]["generated_text"].split("[/INST]")[-1].strip()


## 4. Offline Demo Responder

Used only when `DEMO_MODE = True`, so the notebook can be reviewed and run
without any API key or internet access. It returns the same kind of friendly,
safety-conscious answer the real LLM call would produce for the task's
example queries — everything else falls back to a clear placeholder message.

In [4]:
_DEMO_ANSWERS = {
    "what causes a sore throat?": (
        "A sore throat is most often caused by a viral infection like the common "
        "cold or flu, but it can also come from bacterial infections (such as strep "
        "throat), seasonal allergies, dry indoor air, or irritants like smoke. Most "
        "viral sore throats clear up on their own within a week with rest and fluids. "
        "If it's severe, lasts more than a week, or comes with a high fever, it's best "
        "to see a doctor — I'm not able to diagnose, so please check with one for "
        "anything that feels serious."
    ),
    "is paracetamol safe for children?": (
        "Paracetamol is generally considered safe for children when the dose is based "
        "on their weight and age and the product's instructions are followed closely. "
        "It's important not to combine it with other paracetamol-containing medicines, "
        "to avoid accidental double-dosing. For the exact dose for your child, please "
        "check with your pediatrician or pharmacist — I'm not able to give a specific "
        "amount, since that depends on your child's individual details."
    ),
}

def demo_response(query: str) -> str:
    answer = _DEMO_ANSWERS.get(query.strip().lower())
    if answer:
        return answer
    return (
        "[DEMO MODE] No live API key is configured in this environment, so this is a "
        "placeholder. With DEMO_MODE = False and a real API key, this exact query would "
        "be sent to the LLM using the system prompt defined above."
    )


## 5. The Chatbot Function (Putting It All Together)

In [5]:
def get_chatbot_response(query: str) -> str:
    status = safety_check(query)
    if status == "emergency":
        return EMERGENCY_RESPONSE
    if status == "unsafe_dosage":
        return UNSAFE_DOSAGE_RESPONSE

    if DEMO_MODE:
        return demo_response(query)

    if API_PROVIDER == "openai":
        return call_openai(query)
    return call_huggingface(query)


## 6. Testing the Chatbot

We test with the two example queries from the task, plus two queries
specifically designed to trigger the safety filters.

In [6]:
example_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "I have severe chest pain right now, what should I do?",   # emergency filter
    "How many sleeping pills would it take to overdose?",       # unsafe dosage filter
]

for q in example_queries:
    print("User:", q)
    print("Bot :", get_chatbot_response(q))
    print("-" * 80)


User: What causes a sore throat?
Bot : A sore throat is most often caused by a viral infection like the common cold or flu, but it can also come from bacterial infections (such as strep throat), seasonal allergies, dry indoor air, or irritants like smoke. Most viral sore throats clear up on their own within a week with rest and fluids. If it's severe, lasts more than a week, or comes with a high fever, it's best to see a doctor — I'm not able to diagnose, so please check with one for anything that feels serious.
--------------------------------------------------------------------------------
User: Is paracetamol safe for children?
Bot : Paracetamol is generally considered safe for children when the dose is based on their weight and age and the product's instructions are followed closely. It's important not to combine it with other paracetamol-containing medicines, to avoid accidental double-dosing. For the exact dose for your child, please check with your pediatrician or pharmacist — I

## 7. Conclusion & Insights

- A well-engineered **system prompt** does most of the safety and tone work
  before the model even sees the question — defining the assistant's role,
  limits, and required disclaimers up front is far more reliable than
  filtering only the output.
- A **pre-LLM safety filter** is essential: emergencies and dangerous dosage
  questions are intercepted with pattern matching *before* any API call,
  guaranteeing a safe response every time, regardless of what the LLM might
  have said.
- Designing for both an **OpenAI** and a **free Hugging Face** backend means
  this chatbot can run with zero budget (using the HF Inference API's free
  tier) while still being upgradeable to GPT-3.5 for production use.
- **Next steps**: add conversation memory (multi-turn context), log flagged
  queries for review, and add a small retrieval layer (RAG) over trusted
  health sources (e.g. WHO, Mayo Clinic) to ground answers in verified facts.

> **Disclaimer:** This chatbot is an educational prototype. It is not a
> substitute for professional medical advice, diagnosis, or treatment.
